# Module 10: Chuyển đổi dữ liệu A/D và D/A
Notebook kiểm tra các con số của module (Tooley Ch.3, Floyd Ch.12). Mỗi phần: câu hỏi, mã, rồi ô "Đọc kết quả" nêu đúng số trong đầu ra.

## 1. DAC trọng số nhị phân
Điện áp ra của DAC Tooley Hình 3.6a với mã 1010, 1111, 0001?

In [1]:
import numpy as np
# DAC trong so nhi phan (Tooley Hinh 3.6a): logic 1 = +5 V, Rf = R, dien tro dau vao R, 2R, 4R, 8R
def dac_weighted(code, n=4, v=5.0):
    bits = [(code >> (n - 1 - k)) & 1 for k in range(n)]
    return -sum(v * b / 2 ** k for k, b in enumerate(bits))           # cong dong dien: -Rf * sum(v*b/R_k)
for c in (0b1010, 0b1111, 0b0001):
    print(format(c, '04b'), round(dac_weighted(c), 4), 'V', '| -0.625 x code =', -0.625 * c)

1010 -6.25 V | -0.625 x code = -6.25
1111 -9.375 V | -0.625 x code = -9.375
0001 -0.625 V | -0.625 x code = -0.625


**Đọc kết quả.** Mã 1010 cho -6,25 V và 1111 cho -9,375 V, khớp Tooley (−6,25 V và −9,375 V); mỗi mã tăng thêm −0,625 V.

## 2. Thang R-2R
Giải hệ phương trình nút của thang và so với công thức $-5\cdot code/8$.

In [2]:
# Thang R-2R (Floyd Hinh 12-29): giai he phuong trinh nut, khong dung cong thuc dong
R = 1000.0
def r2r_nodal(code, n=4, vref=5.0):
    bits = [(code >> (n - 1 - k)) & 1 for k in range(n)]
    m = n - 1; G = np.zeros((m, m)); b = np.zeros(m)
    for k in range(m):
        G[k, k] += 1 / (2 * R); b[k] += bits[1 + k] * vref / (2 * R); G[k, k] += 1 / R
        if k > 0: G[k, k - 1] -= 1 / R
        if k < m - 1: G[k, k] += 1 / R; G[k, k + 1] -= 1 / R
        else: G[k, k] += 1 / (2 * R)
    v = np.linalg.solve(G, b)
    return -(2 * R) * (bits[0] * vref / (2 * R) + v[0] / R)
for c in (8, 4, 2, 1, 0b1010):
    print(format(c, '04b'), round(r2r_nodal(c), 4), 'V', '| cong thuc dong: ', -5 * c / 8)

1000 -5.0 V | cong thuc dong:  -5.0
0100 -2.5 V | cong thuc dong:  -2.5
0010 -1.25 V | cong thuc dong:  -1.25
0001 -0.625 V | cong thuc dong:  -0.625
1010 -6.25 V | cong thuc dong:  -6.25


**Đọc kết quả.** Chỉ một bit bằng 1 cho −5; −2,5; −1,25; −0,625 V (Floyd Hình 12-30) và mã 1010 cho -6,25 V.

## 3. ADC xấp xỉ liên tiếp
Các bước với 5,1 V và mã ra cho 11,7 V.

In [3]:
# ADC xap xi lien tiep 4 bit, trong so 8, 4, 2, 1 V (Floyd Hinh 12-17, 12-18)
def sar(vin, n=4):
    code = 0; trace = []
    for k in range(n):
        trial = code | (1 << (n - 1 - k)); dac = sum(2 ** (n - 1 - j) for j in range(n) if (trial >> (n - 1 - j)) & 1)
        keep = dac <= vin; trace.append((dac, keep))
        if keep: code = trial
    return code, trace
code, trace = sar(5.1); print(format(code, '04b'), trace)
print(format(sar(11.7)[0], '04b'))

0101 [(8, False), (4, True), (6, False), (5, True)]
1011


**Đọc kết quả.** 5,1 V: 8 V bỏ, 4 V giữ, 6 V bỏ, 5 V giữ, mã 0101 (Floyd). 11,7 V cho mã 1011.

## 4. ADC flash 3 bit
Mã ra cho 0,3; 2,0; 4,7; 6,9 V.

In [4]:
# ADC flash 3 bit (Tooley Hinh 3.9): nguong 0,5; 1,5; ...; 6,5 V, bo ma hoa uu tien
def flash(vin, n=3):
    outs = [1 if vin > k + 0.5 else 0 for k in range(2 ** n - 1)]
    return max([k + 1 for k, o in enumerate(outs) if o], default=0), outs
for v in (0.3, 2.0, 4.7, 6.9): print(v, format(flash(v)[0], '03b'))

0.3 000
2.0 010
4.7 101
6.9 111


**Đọc kết quả.** Vào 2 V cho 010 như Tooley; 6,9 V vượt mọi ngưỡng nên cho 111.

## 5. Độ phân giải, bộ so sánh, Nyquist
Độ phân giải của DAC n bit và số bộ so sánh của flash.

In [5]:
# Do phan giai, so bo so sanh, Nyquist
for n in (4, 8, 12, 16): print(n, 'bit: do phan giai', round(100 / (2 ** n - 1), 5), '%', '| flash can', 2 ** n - 1, 'bo so sanh')
fmax = 15e3; print('fs toi thieu cho', fmax / 1e3, 'kHz:', 2 * fmax / 1e3, 'kHz')
f, fs = 5.0, 8.0; print('alias cua', f, 'kHz lay mau', fs, 'kHz:', abs(f - fs * round(f / fs)), 'kHz')

4 bit: do phan giai 6.66667 % | flash can 15 bo so sanh
8 bit: do phan giai 0.39216 % | flash can 255 bo so sanh
12 bit: do phan giai 0.02442 % | flash can 4095 bo so sanh
16 bit: do phan giai 0.00153 % | flash can 65535 bo so sanh
fs toi thieu cho 15.0 kHz: 30.0 kHz
alias cua 5.0 kHz lay mau 8.0 kHz: 3.0 kHz


**Đọc kết quả.** 8 bit: 0,39216 % và 255 bộ so sánh; 12 bit: 0,02442 %; 15 kHz cần ít nhất 30 kHz; sóng 5 kHz lấy mẫu 8 kHz hiện thành 3 kHz.